训练代码

In [ ]:
#coding:utf-8
import os
import sys
from ultralytics import YOLO
import torch
import warnings

warnings.filterwarnings('ignore')

# 🔥 小麦病虫害项目根目录（固定）
ROOT_DIR = r"D:\微信QQ\FireDetection"
os.chdir(ROOT_DIR)

def get_abs_path(relative_path):
    # 直接用根目录拼接
    abs_path = os.path.join(ROOT_DIR, relative_path)
    return os.path.normpath(abs_path)

def check_file_exists(file_path):
    if os.path.exists(file_path):
        print(f"✅ 找到文件：{file_path}")
        return True
    else:
        print(f"❌ 未找到文件：{file_path}")
        return False

def print_gpu_memory_info():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1e6
        reserved = torch.cuda.memory_reserved(0) / 1e6
        max_allocated = torch.cuda.max_memory_allocated(0) / 1e6

        print(f"\n📊 GPU内存使用情况：")
        print(f"  已分配: {allocated:.0f} MB")
        print(f"  已预留: {reserved:.0f} MB")
        print(f"  最大分配: {max_allocated:.0f} MB")
        torch.cuda.empty_cache()
    else:
        print("\n⚠️  未检测到GPU，将使用CPU训练")

def train_xiaomai_disease():
    print(f"\n📌 小麦病虫害检测训练")
    print(f"📌 项目根目录：{os.getcwd()}")

    # 数据集路径
    data_yaml_abs = get_abs_path("datasets\data.yaml")

    if not check_file_exists(data_yaml_abs):
        print("\n❌ 数据集配置文件不存在，退出训练")
        sys.exit(1)

    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"\n📌 使用设备：{device}")

    print("\n🔧 加载YOLOv8n预训练模型...")
    model = YOLO("yolov8n.pt")

    print("\n📊 初始显存使用：")
    print_gpu_memory_info()

    # 训练参数
    train_config = {
        'data': data_yaml_abs,
        'epochs': 10,#训练轮数
        'batch': 16,
        'imgsz': 640,
        'device': device,
        'augment': True,
        'degrees': 60,
        'fliplr': 0.5,
        'flipud': 0.3,
        'scale': 0.6,
        'translate': 0.15,
        'perspective': 0.001,
        'patience': 10,
        'workers': 0 if device == 'cpu' else 2,
        'save': True,
        'cache': 'ram',
        'amp': torch.cuda.is_available(),
        'verbose': True,
        'project': 'runs/detect',
        'name': 'xiaomai_model',
        'exist_ok': True,
    }

    print("\n🚀 开始训练小麦病虫害检测模型...")
    try:
        results = model.train(**train_config)
        print("\n✅ 训练完成！")

        # 导出 ONNX
        print("\n🔄 导出模型为 ONNX 格式...")
        model.export(format='onnx', device='cpu', imgsz=640, simplify=True)
        print("✅ ONNX 导出成功！")

    except Exception as e:
        print(f"\n❌ 训练出错：{str(e)[:150]}")

if __name__ == '__main__':
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(42)

    train_xiaomai_disease()
    print("\n🎉 小麦病虫害模型训练全部完成！")

摄像头与华为云同步代码

In [ ]:
# 🔥 Jupyter 必须加这一行，才能显示图像
%matplotlib inline

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np
import time
import json
import paho.mqtt.client as mqtt
from paho.mqtt.client import CallbackAPIVersion   # 使用 VERSION2 API

# ==================== 强制中文正常显示 ====================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)

# ==================== 华为云 IoT 设备信息====================
CLIENT_ID = "6a44f1107f2e6c302f80df88_farmeye-1_0_0_2026070110"  
USERNAME  = "6a44f1107f2e6c302f80df88_farmeye-1"
PASSWORD  = "76fb0e768bbeb4f53f4b38b9dab54d4dadb833b3f1d1fecc6a2f8b36afdffeb6"
HOST      = "a963d886c0.st1.iotda-device.cn-north-4.myhuaweicloud.com"
PORT      = 1883

# 属性上报主题
TOPIC_PROPERTY = f"$oc/devices/{CLIENT_ID}/sys/properties/report"
# 告警事件上报主题
TOPIC_ALERT    = f"$oc/devices/{CLIENT_ID}/sys/events/up"

# ==================== 模型路径 ====================
model_path = r'D:\微信QQ\FireDetection\runs\detect\runs\detect\xiaomai_model\weights\best.pt'
model = YOLO(model_path, task='detect')

# ==================== 打开摄像头 ====================
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("摄像头打开失败")
    exit()

# ==================== MQTT 连接函数（与模拟器相同的 VERSION2）====================
def connect_mqtt():
    def on_connect(client, userdata, flags, rc):
        if rc == 0:
            print("✅ 华为云IoT连接认证成功！")
        else:
            print(f"❌ 连接失败，错误码：{rc}")

    def on_disconnect(client, userdata, rc):
        print("⚠️ 与云端断开连接，尝试重连...")
        client.reconnect()

    client = mqtt.Client(CallbackAPIVersion.VERSION2, CLIENT_ID)
    client.username_pw_set(USERNAME, PASSWORD)
    client.on_connect = on_connect
    client.on_disconnect = on_disconnect
    client.connect(HOST, PORT)
    return client

client = connect_mqtt()
client.loop_start()

print("摄像头已启动 → 按 q 键停止")

# ==================== 辅助函数 ====================
def build_properties(results):
    """构造属性上报 JSON"""
    objs = []
    class_names = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = model.names[cls_id]
        class_names.append(class_name)
        objs.append({
            "类别": class_name,
            "置信度": round(conf, 2),
            "位置": [round(x, 2) for x in box.xyxy[0].tolist()]
        })
    return {
        "services": [{
            "service_id": "farmeye_ai",
            "properties": {
                "crop_type": "wheat",
                "disease_type": ", ".join(set(class_names)) if class_names else "",
                "object_number": len(objs),
                "max_conf": max([o["置信度"] for o in objs]) if objs else 0,
                "all_object": objs,
                "timestamp": time.time()
            }
        }]
    }

def build_alert_event(results):
    """构造告警事件 JSON"""
    objs = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = model.names[cls_id]
        objs.append({
            "病害类型": class_name,
            "置信度": round(conf, 2)
        })
    return {
        "services": [{
            "service_id":"farmeye_ai",
            "event_time": time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()),
            "event_id": "pest_detected",
            "event_type": "warning",
            "paras": {
                "目标列表": objs,
                "最高置信度": max([o["置信度"] for o in objs]) if objs else 0
            }
        }]
    }

# ==================== 上报控制变量 ====================
last_alert_time = 0
ALERT_COOLDOWN = 2       # 2 秒内不重复告警

last_upload_time = 0
UPLOAD_INTERVAL = 3     # 属性上报间隔 3 秒

# ==================== 实时检测循环 ====================
try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("读取失败")
            break

        # YOLO 检测
        results = model(frame)
        frame_det = results[0].plot()

        current_time = time.time()

        # ------ 属性上报（每 3 秒）------
        if current_time - last_upload_time >= UPLOAD_INTERVAL:
            payload = build_properties(results)
            print(json.dumps(payload, ensure_ascii=False, indent=2))
            client.publish(TOPIC_PROPERTY, json.dumps(payload, ensure_ascii=False), qos=1)
            print(f"📡 属性上报：{json.dumps(payload, ensure_ascii=False, indent=2)}")
            last_upload_time = current_time

        # ------ 告警事件上报（检测到目标，且冷却时间已过）------
        if len(results[0].boxes) > 0 and (current_time - last_alert_time) >= ALERT_COOLDOWN:
            alert_json = build_alert_event(results)
            client.publish(TOPIC_ALERT, json.dumps(alert_json, ensure_ascii=False), qos=1)
            print(f"🚨 告警事件已推送：{json.dumps(alert_json, ensure_ascii=False, indent=2)}")
            last_alert_time = current_time

        # 显示画面
        frame_rgb = cv2.cvtColor(frame_det, cv2.COLOR_BGR2RGB)
        clear_output(wait=True)
        plt.imshow(frame_rgb)
        plt.title("小麦病虫害实时检测 - 已接入华为云", fontsize=16)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    client.loop_stop()
    client.disconnect()
    print("已停止检测，与云端连接已关闭")